# 05 · Consequence: system rankings (Section 5.3)

A drop in agreement matters most when it changes which system wins. Because the human scores are identical in both scripts, every change in ranking comes from the metric.

* A *flip* is a pair of systems whose order changes between the native and the romanised scores.
* *acc\*eq* is the share of translation pairs the metric orders as humans do, with ties handled fairly.

This notebook only reads tables written by notebooks 03 and 04.

In [ ]:
import os
import sys
import warnings
from pathlib import Path

# quieten library notices that do not affect any result
os.environ["TRANSFORMERS_VERBOSITY"] = "error"      # "no PyTorch found": no model is needed here
warnings.filterwarnings("ignore", module="tqdm")
warnings.filterwarnings("ignore", module="transformers")

import pandas as pd

ROOT = Path.cwd().resolve().parent          # the repository root
sys.path.insert(0, str(ROOT / "src"))
TABLES = ROOT / "results" / "tables"
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 3)

def table(name):
    """A table written by the analysis, from results/tables/."""
    return pd.read_csv(TABLES / f"{name}.csv")

## IndicMT Eval: system pairs that flip (6 systems, so 15 pairs per language, 75 in all)

In [ ]:
f = table("system_ranking_flips")
f = f[f.scheme == "raw"]                       # the metric's own scores (scheme "qn" is after rescaling, notebook 07)
f.groupby("metric").n_flips.sum().sort_values(ascending=False).rename("flips of 75")

## Pairwise accuracy hides much of this

COMET-22's acc\*eq falls only from 0.611 to 0.550, although it reorders systems. An uninformative metric scores 0.442 here.

In [ ]:
a = table("metaeval_acc_star_eq")      # one row per metric, language and condition
# pooled over the five languages, each weighted by its number of translation pairs
a["w"] = a.acc_star_eq * a.n_pairs
g = a.groupby(["metric", "condition"])[["w", "n_pairs"]].sum()
(g.w / g.n_pairs).unstack().round(3)

## WMT: flips on the compared metric and setting pairs

In [ ]:
m = table("beyond_indic_metaeval")
m = m[m.compared]
m.groupby("metric")[["n_flips", "n_pairs"]].sum().assign(pct=lambda x: (100 * x.n_flips / x.n_pairs).round(1))